In [1]:
import requests
import sqlite3

In [2]:
BASE_URL = "https://api-v3.monarchinitiative.org/v3/api"

def executar_pipeline_monarch():
    print("1. A pesquisar genes na Monarch API...")
    url_pesquisa = f"{BASE_URL}/search"
    params_pesquisa = {"q": "*", "category": "biolink:Gene", "limit": 120}
    
    resposta_pesquisa = requests.get(url_pesquisa, params=params_pesquisa).json()
    items = resposta_pesquisa.get('items', [])
    
    # Guardar apenas os genes que começam por 'HGNC:'
    genes_hgnc = [i for i in items if i.get('id', '').startswith('HGNC:')]
    
    dados_para_ingestao = []
    meta_registos = 45  # Queremos recolher entre 40 a 50 registos
    
    print("2. A recolher associações causais Gene-Doença da API...")
    for gene in genes_hgnc:
        if len(dados_para_ingestao) >= meta_registos:
            break
            
        gene_id = gene['id']
        gene_name = gene.get('title', 'Unknown_Gene')
        
        # Fazer a requisição de associações para este gene específico
        url_assoc = f"{BASE_URL}/entity/{gene_id}/biolink:CausalGeneToDiseaseAssociation"
        resposta_assoc = requests.get(url_assoc, params={"limit": 10}).json()
        
        for assoc in resposta_assoc.get('items', []):
            if len(dados_para_ingestao) >= meta_registos:
                break
                
            dados_para_ingestao.append({
                'gene_id': gene_id,
                'gene_name': gene_name,
                'disease_id': assoc.get('object'),
                'disease_name': assoc.get('object_label', 'Unknown_Disease'),
                'score': assoc.get('score', None)
            })

    # 3. Processo de gravação na Base de Dados DisGeNET
    print(f"3. A iniciar escrita de {len(dados_para_ingestao)} registos na base de dados...")
    conn = sqlite3.connect('disgenet.db')
    cursor = conn.cursor()
    
    try:
        # Garantir a criação das tabelas com os nomes conceituais exigidos neste exercício
        cursor.execute("CREATE TABLE IF NOT EXISTS DIM_Gene (geneID TEXT PRIMARY KEY, geneName TEXT);")
        cursor.execute("CREATE TABLE IF NOT EXISTS DIM_Disease (diseaseID TEXT PRIMARY KEY, diseaseName TEXT, typeID TEXT);")
        cursor.execute("CREATE TABLE IF NOT EXISTS Fact_GDA (geneID TEXT, diseaseID TEXT, year INTEGER, nPmid TEXT, score REAL);")
        
        for r in dados_para_ingestao:
            # Popular as dimensões primeiro (evitando duplicar chaves primárias)
            cursor.execute("INSERT OR IGNORE INTO DIM_Gene (geneID, geneName) VALUES (?, ?)", (r['gene_id'], r['gene_name']))
            cursor.execute("INSERT OR IGNORE INTO DIM_Disease (diseaseID, diseaseName, typeID) VALUES (?, ?, NULL)", (r['disease_id'], r['disease_name']))
            
            # Inserir na tabela de Factos (Regras: Ano=2026, nPmid=NULL)
            cursor.execute("""
                INSERT INTO Fact_GDA (geneID, diseaseID, year, nPmid, score) 
                VALUES (?, ?, 2026, NULL, ?)
            """, (r['gene_id'], r['disease_id'], r['score']))
            
        conn.commit()
        print("Pipeline finalizado! O teu Cubo de Dados foi enriquecido com sucesso.")
        
    except Exception as e:
        conn.rollback()
        print(f"Erro crítico detetado no pipeline. Transação abortada: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    executar_pipeline_monarch()

1. A pesquisar genes na Monarch API...
2. A recolher associações causais Gene-Doença da API...
3. A iniciar escrita de 5 registos na base de dados...
Pipeline finalizado! O teu Cubo de Dados foi enriquecido com sucesso.
